# Lab 4: Knowledge e RAG (15 min)

## Objetivos
- Entender o padrão RAG (Retrieval-Augmented Generation)
- Criar embeddings com `azure-ai-inference`
- Pesquisar com Azure AI Search
- Combinar pesquisa + geração num pipeline RAG

## Conceitos

### O que é RAG?
```
Pergunta do utilizador
        │
        ▼
┌─────────────────┐
│  1. Embedding    │  Converter pergunta em vector
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  2. Search       │  Encontrar documentos relevantes
│  (AI Search)     │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  3. Generate     │  Modelo gera resposta com contexto
│  (LLM + docs)   │
└─────────────────┘
```

### Porquê RAG?
- O modelo não conhece os **teus dados**
- RAG adiciona **contexto relevante** ao prompt
- Mais preciso e factual que o modelo sozinho

In [ ]:
# Setup
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from azure.ai.inference import ChatCompletionsClient, EmbeddingsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
key = os.getenv("AZURE_AI_FOUNDRY_KEY")
model = os.getenv("MODEL_DEPLOYMENT", "gpt-4o")
embedding_model = os.getenv("EMBEDDING_DEPLOYMENT", "text-embedding-ada-002")

# Clientes Foundry
chat_client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key),
)

embeddings_client = EmbeddingsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key),
)

print(f"✅ Clientes criados")
print(f"   Chat: {model}")
print(f"   Embeddings: {embedding_model}")

## 🖥️ Exercício 4.1: Criar Embeddings

Embeddings convertem texto em vectores numéricos que representam o significado semântico.

In [ ]:
# Exercício 4.1: Criar embeddings e medir semelhança
import numpy as np

textos = [
    "O Azure é uma plataforma de cloud computing da Microsoft",
    "A Microsoft oferece serviços de computação na nuvem com o Azure",
    "O meu gato gosta de dormir ao sol durante a tarde",
]

response = embeddings_client.embed(
    model=embedding_model,
    input=textos,
)

vectors = [item.embedding for item in response.data]
print(f"✅ {len(vectors)} embeddings criados (dimensão: {len(vectors[0])})")

# Calcular similaridade coseno
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"\n📊 Semelhança entre textos:")
print(f"   Texto 1 vs 2 (semelhantes): {cosine_similarity(vectors[0], vectors[1]):.4f}")
print(f"   Texto 1 vs 3 (diferentes):  {cosine_similarity(vectors[0], vectors[2]):.4f}")
print(f"   Texto 2 vs 3 (diferentes):  {cosine_similarity(vectors[1], vectors[2]):.4f}")

## 🖥️ Exercício 4.2: RAG Simples (sem AI Search)

Vamos implementar RAG localmente usando os nossos documentos de exemplo.

In [ ]:
# Exercício 4.2: RAG local com documentos de exemplo

# Carregar documento
with open("../data/documentos/exemplo.md", "r", encoding="utf-8") as f:
    documento = f.read()

# Dividir em chunks
paragrafos = [p.strip() for p in documento.split("\n\n") if p.strip() and len(p.strip()) > 20]
print(f"📄 Documento dividido em {len(paragrafos)} chunks")

# Criar embeddings dos chunks
resp = embeddings_client.embed(
    model=embedding_model,
    input=paragrafos,
)
chunk_vectors = [item.embedding for item in resp.data]

# Função de pesquisa
def pesquisar(pergunta: str, top_k: int = 3) -> list:
    q_resp = embeddings_client.embed(model=embedding_model, input=[pergunta])
    q_vec = q_resp.data[0].embedding

    scores = [(i, cosine_similarity(q_vec, cv)) for i, cv in enumerate(chunk_vectors)]
    scores.sort(key=lambda x: x[1], reverse=True)

    return [(paragrafos[i], score) for i, score in scores[:top_k]]

# Testar pesquisa
resultados = pesquisar("política de férias")
print(f"\n🔍 Resultados para 'política de férias':")
for texto, score in resultados:
    print(f"   [{score:.3f}] {texto[:80]}...")

In [ ]:
# Pipeline RAG completo
def rag(pergunta: str) -> str:
    """Pipeline RAG: pesquisa + geração."""
    # 1. Pesquisar documentos relevantes
    resultados = pesquisar(pergunta, top_k=3)
    contexto = "\n\n".join([texto for texto, _ in resultados])

    # 2. Gerar resposta com contexto
    response = chat_client.complete(
        model=model,
        messages=[
            SystemMessage(content="Responde à pergunta baseando-te APENAS no contexto fornecido. Se a informação não estiver no contexto, diz que não tens essa informação. Responde em português."),
            UserMessage(content=f"Contexto:\n{contexto}\n\nPergunta: {pergunta}"),
        ],
        max_tokens=300,
        temperature=0.3,
    )

    return response.choices[0].message.content

# Testar RAG
perguntas = [
    "Quantos dias de férias tem um funcionário?",
    "Qual é a política de trabalho remoto?",
    "Qual é o capital de Portugal?",  # Não está no documento!
]

for p in perguntas:
    print(f"\n👤 {p}")
    print(f"🤖 {rag(p)}")
    print("-" * 60)

## 🖥️ Exercício 4.3: RAG com Azure AI Search (Opcional)

Se tiveres AI Search configurado, podes usar pesquisa vetorial + semântica.

> ⚠️ Este exercício requer `AZURE_SEARCH_ENDPOINT` e `AZURE_SEARCH_KEY` no `.env`

In [ ]:
# Exercício 4.3 (Opcional): RAG com AI Search
search_endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
search_key = os.getenv("AZURE_SEARCH_KEY")

if search_endpoint and search_key and not search_endpoint.startswith("<"):
    from azure.search.documents import SearchClient
    from azure.search.documents.models import VectorizedQuery

    search_client = SearchClient(
        endpoint=search_endpoint,
        index_name="workshop-index",
        credential=AzureKeyCredential(search_key),
    )

    # Pesquisa híbrida (texto + vector)
    pergunta = "política de férias"
    q_resp = embeddings_client.embed(model=embedding_model, input=[pergunta])
    q_vec = q_resp.data[0].embedding

    results = search_client.search(
        search_text=pergunta,
        vector_queries=[VectorizedQuery(vector=q_vec, k_nearest_neighbors=3, fields="contentVector")],
        top=3,
    )

    print("🔍 Resultados do AI Search:")
    for r in results:
        print(f"   [{r['@search.score']:.3f}] {r['content'][:80]}...")
else:
    print("⏭️ AI Search não configurado - exercício saltado.")
    print("   Para ativar, configura AZURE_SEARCH_ENDPOINT e AZURE_SEARCH_KEY no .env")

## ✅ Resumo

Neste lab aprendeste:
- O padrão **RAG** (Retrieval-Augmented Generation)
- Criar **embeddings** com `EmbeddingsClient` do Foundry
- Implementar **pesquisa semântica** com similaridade coseno
- Construir um **pipeline RAG completo**
- (Opcional) Usar **Azure AI Search** para pesquisa vetorial

**Próximo:** [Lab 5 - Workflows](lab05-workflows.ipynb)